# Global Shark Attacks Analysis & International Tourism Risk Correlation

**Course:** Programming for Data Science — ESILV (Master Data & IA, A4 DIA3, 2026-2027)  
**Instructor:** Hugo Alatrista-Salas  
**GitHub Repository:** https://github.com/groupe-dia3-ml/programming-for-data-science  

### Team Members
- **Maxime Laurent** (Team Leader) — `@maximelrnt`
- **Louis Le Forestier** — `@louislefo`
- **Baptiste Noailhac** — `@BaptisteNoai`
- **Romain Thepaut** — `@RomThpt`

### Multidimensional Datasets Used
1. **Primary Dataset (Multidimensional):** *Global Shark Attack File (GSAF)* — https://www.kaggle.com/datasets/mexwell/global-shark-attack  
   - **Temporal:** `Date`, `Year`, `Month`, `Time` (filtered to reliable modern reporting: 1960-2018).
   - **Spatial:** `Country`, `Area`, `Location` enriched with coastal coordinates `(latitude, longitude)` and Natural Earth country polygons.
   - **Analytical:** `Type`, `Activity`, `Species`, victim `Sex`/`Age`, `Injury`, and `Fatal (Y/N)`.
2. **External Bonus Dataset (+1 Extra Point):** *World Bank International Tourism: Number of Arrivals (`ST.INT.ARVL`)* — https://data.worldbank.org/indicator/ST.INT.ARVL  
   - Tracks annual inbound international tourist arrivals across 266 countries/regions (**1995-2020**), joined on `(Country_clean, Year)` to measure risk per 1 million tourists.

*Note: This deliverable notebook is **100% self-contained** — all functions (`# Function / # Input / # Output`) are defined directly inside this file and executed from the final `if __name__ == '__main__':` block.*

In [1]:
# Team members and GitHub link
# Maxime, Laurent (team head)
# Louis, Le Forestier (team member)
# Baptiste, Noailhac (team member)
# Romain, Thepaut (team member)
# Link to GitHub repository: https://github.com/groupe-dia3-ml/programming-for-data-science

# Import libraries
import io
import os
import re
import zipfile
import warnings
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import pearsonr, spearmanr
from mlxtend.frequent_patterns import apriori, association_rules
from prophet import Prophet
from sklearn.cluster import OPTICS, KMeans
import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc

warnings.filterwarnings('ignore')
print('All libraries imported successfully.')

All libraries imported successfully.


## Step 1: Data Loading, EDA Profiling & Cleaning Functions `[2 points]`

### Domain Meaning & Preprocessing Justification
- **Dropped Columns:** `href`, `href formula`, `pdf`, `Case Number.1`, `Case Number.2`, `original order`, `Unnamed: 22`, `Unnamed: 23`, and `Investigator or Source` are removed because they are administrative links or blank CSV export artifacts.
- **Blank Padding Rows:** Raw `attacks.csv` contains over 17,000 empty trailing lines that are dropped.
- **Modern Period (`1960 <= Year <= 2023`):** Removes sparse historical entries with missing dates and reporting bias.
- **Feature Engineering:** Normalizes `Country` names to match World Bank records (`Country_clean`), extracts `Month` and hemisphere-aware `Season`, and categorizes `Activity`, `Species`, `Time`, and `Fatal (Y/N)` into clean discrete attributes.

In [2]:
# Official / public direct download URLs for cross-platform automatic fallback (Windows, macOS, Linux)
SHARK_FALLBACK_URL = "https://raw.githubusercontent.com/SmilodonCub/DATA607/master/attacks.csv"
TOURISM_FALLBACK_URL = "https://api.worldbank.org/v2/en/indicator/ST.INT.ARVL?downloadformat=csv"


# Mapping dictionary to align GSAF country names with World Bank official names
COUNTRY_MAPPING = {
    "USA": "United States",
    "UNITED STATES": "United States",
    "ENGLAND": "United Kingdom",
    "SCOTLAND": "United Kingdom",
    "WALES": "United Kingdom",
    "UNITED KINGDOM": "United Kingdom",
    "BAHAMAS": "Bahamas, The",
    "EGYPT": "Egypt, Arab Rep.",
    "RUSSIA": "Russian Federation",
    "SOUTH KOREA": "Korea, Rep.",
    "IRAN": "Iran, Islamic Rep.",
    "VENEZUELA": "Venezuela, RB",
    "HONG KONG": "Hong Kong SAR, China",
    "TURKEY": "Turkiye",
    "VIETNAM": "Viet Nam",
    "ST. MARTIN": "St. Martin (French part)",
    "REUNION": "Reunion",
    "BERMUDA": "Bermuda",
    "NEW CALEDONIA": "New Caledonia",
    "PAPUA NEW GUINEA": "Papua New Guinea",
    "FIJI": "Fiji",
    "FRENCH POLYNESIA": "French Polynesia",
}


# Function for finding a dataset file across standard local folders (Windows/macOS/Linux safe)
# Input: file_path (str or Path) - requested path to the dataset file
# Output: Path object pointing to an existing local file, or None if not found locally
def find_local_data_file(file_path):
    requested = Path(file_path)
    filename = requested.name
    candidates = [
        requested,
        Path("..") / "projet" / "data" / filename,
        Path("..") / "data" / filename,
        Path("projet") / "data" / filename,
        Path("data") / filename,
        Path(filename),
    ]
    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate
    return None


# Function for loading the primary Global Shark Attack dataset (local first, auto-download fallback)
# Input: file_path (str or Path) - path to attacks.csv or GSAF5.xls
# Output: raw pandas DataFrame of shark attacks
def load_shark_data(file_path):
    local_file = find_local_data_file(file_path)

    if local_file is None:
        # Cross-platform fallback: download directly into local ./data/ folder (no Kaggle account needed)
        target = Path("data") / "attacks.csv"
        target.parent.mkdir(parents=True, exist_ok=True)
        print(f"Local shark dataset not found. Downloading automatically to {target} ...")
        response = requests.get(SHARK_FALLBACK_URL, timeout=30)
        response.raise_for_status()
        target.write_bytes(response.content)
        local_file = target
        print(f"Downloaded shark dataset successfully ({local_file.stat().st_size:,} bytes).")

    # Detect file format automatically so both CSV and Excel exports work on any OS
    if local_file.suffix.lower() in (".xls", ".xlsx"):
        df = pd.read_excel(local_file)
    else:
        # Standard GSAF CSV uses latin-1 encoding due to accented place names
        df = pd.read_csv(local_file, encoding="latin-1", low_memory=False)

    # Strip trailing spaces from column names (e.g. 'Sex ' -> 'Sex', 'Species ' -> 'Species')
    df.columns = df.columns.str.strip()
    return df


# Function for loading the external World Bank tourism dataset (local first, auto-download fallback)
# Input: file_path (str or Path) - path to tourism_arrivals.csv
# Output: raw pandas DataFrame of international tourist arrivals
def load_tourism_data(file_path):
    local_file = find_local_data_file(file_path)

    if local_file is None:
        # Cross-platform fallback: download and extract official World Bank ST.INT.ARVL CSV ZIP in memory
        target = Path("data") / "tourism_arrivals.csv"
        target.parent.mkdir(parents=True, exist_ok=True)
        print(f"Local tourism dataset not found. Downloading World Bank ST.INT.ARVL to {target} ...")
        response = requests.get(TOURISM_FALLBACK_URL, timeout=30)
        response.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            csv_members = [name for name in zf.namelist() if name.startswith("API_ST.INT.ARVL")]
            if not csv_members:
                raise RuntimeError("Could not locate API_ST.INT.ARVL CSV inside World Bank ZIP archive.")
            target.write_bytes(zf.read(csv_members[0]))
        local_file = target
        print(f"Downloaded World Bank tourism dataset successfully ({local_file.stat().st_size:,} bytes).")

    # World Bank CSV downloads start with 4 metadata lines before the header
    df = pd.read_csv(local_file, skiprows=4)
    df.columns = df.columns.str.strip()
    # Remove any completely empty trailing column
    df = df.dropna(how="all", axis=1)
    return df


# Function for profiling and summarizing any dataset (mandatory EDA step)
# Input: df (pd.DataFrame), dataset_name (str)
# Output: dictionary with shape, dtypes, missing values, numeric ranges, and categorical counts
def summarize_dataset(df, dataset_name="Dataset"):
    summary = {
        "dataset_name": dataset_name,
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
        "columns": list(df.columns),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "missing_counts": df.isna().sum().to_dict(),
        "missing_pct": (df.isna().mean() * 100).round(2).to_dict(),
        "numeric_ranges": {},
        "categorical_unique": {},
    }

    # Compute min, max, mean, and median for numeric columns
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        series = df[col].dropna()
        if len(series) > 0:
            summary["numeric_ranges"][col] = {
                "min": float(series.min()),
                "max": float(series.max()),
                "mean": round(float(series.mean()), 2),
                "median": round(float(series.median()), 2),
            }

    # Compute cardinality (unique values) for categorical columns
    cat_cols = df.select_dtypes(exclude=[np.number]).columns
    for col in cat_cols:
        summary["categorical_unique"][col] = int(df[col].nunique(dropna=True))

    return summary


# Function for grouping noisy free-text Activity descriptions into clean categories
# Input: act (str or NaN) - raw activity text from GSAF
# Output: standardized activity category string
def categorize_activity(act):
    if not isinstance(act, str) or pd.isna(act):
        return "Other"
    act_lower = act.lower()
    if any(k in act_lower for k in ["surf", "bodyboard", "paddleboard", "boogie", "kite"]):
        return "Surfing"
    if "snork" in act_lower or any(k in act_lower for k in ["div", "spear"]):
        return "Diving/Snorkeling"
    if any(k in act_lower for k in ["swim", "bath", "float"]):
        return "Swimming"
    if "fish" in act_lower:
        return "Fishing"
    if any(k in act_lower for k in ["wading", "standing", "treading", "walking"]):
        return "Wading"
    if any(k in act_lower for k in ["boat", "kayak", "canoe", "rowing", "sailing", "ski"]):
        return "Boating/Kayaking"
    return "Other"


# Function for extracting major shark species groups from raw Species text
# Input: species_val (str or NaN) - raw shark species description
# Output: standardized shark species group string
def categorize_species(species_val):
    if not isinstance(species_val, str) or pd.isna(species_val):
        return "Unidentified"
    s = species_val.lower().strip()
    if s == "" or any(k in s for k in ["unconfirmed", "not confirmed", "questionable", "invalid", "unidentified"]):
        return "Unidentified"
    if "white" in s:
        return "White Shark"
    if "tiger" in s:
        return "Tiger Shark"
    if "bull" in s or "zambezi" in s:
        return "Bull Shark"
    if "blacktip" in s or "spinner" in s:
        return "Blacktip/Spinner Shark"
    if "reef" in s or "nurse" in s or "lemon" in s or "wobbegong" in s:
        return "Reef/Nurse Shark"
    if "hammerhead" in s or "mako" in s or "bronze" in s or "whaler" in s or "raggedtooth" in s or "blue" in s:
        return "Hammerhead/Mako/Whaler"
    if "shark" in s:
        return "Other Named Shark"
    return "Unidentified"


# Function for extracting month number (1-12) from unstructured Date strings
# Input: date_val (str or NaN) - raw GSAF Date string
# Output: integer month (1 to 12) or np.nan if missing
def parse_month(date_val):
    if not isinstance(date_val, str) or pd.isna(date_val):
        return np.nan
    months = {
        "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
        "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
    }
    date_lower = date_val.lower()
    for m_str, m_num in months.items():
        if m_str in date_lower:
            return m_num
    return np.nan


# Function for assigning hemisphere-aware Season from Month and Country
# Input: month (float/int), country (str)
# Output: Season string ('Summer', 'Autumn', 'Winter', 'Spring', or 'Unknown')
def assign_season(month, country):
    if pd.isna(month):
        return "Unknown"
    month = int(month)
    southern_countries = {
        "Australia", "South Africa", "New Zealand", "Brazil", "Chile",
        "Argentina", "Uruguay", "Reunion", "Fiji", "New Caledonia",
        "Papua New Guinea", "French Polynesia", "Mozambique",
    }
    is_south = country in southern_countries

    if month in (12, 1, 2):
        return "Summer" if is_south else "Winter"
    if month in (3, 4, 5):
        return "Autumn" if is_south else "Spring"
    if month in (6, 7, 8):
        return "Winter" if is_south else "Summer"
    if month in (9, 10, 11):
        return "Spring" if is_south else "Autumn"
    return "Unknown"


# Function for bucketing raw incident Time text into Time_of_day periods
# Input: t_val (str or NaN) - raw time string (e.g. '14h30', 'Afternoon')
# Output: period label ('Morning (06-12)', 'Afternoon (12-18)', 'Evening/Night (18-06)', 'Unknown')
def categorize_time(t_val):
    if not isinstance(t_val, str) or pd.isna(t_val):
        return "Unknown"
    t_val = t_val.lower().strip()
    if any(k in t_val for k in ["afternoon", "midday", "noon"]):
        return "Afternoon (12-18)"
    if "morning" in t_val:
        return "Morning (06-12)"
    if any(k in t_val for k in ["night", "evening", "dusk", "sunset"]):
        return "Evening/Night (18-06)"

    # Parse numeric hour formats such as 14h00, 14:30, or 1400
    match = re.search(r"(\d{1,2})[h:]?(\d{2})?", t_val)
    if match:
        hour = int(match.group(1))
        if 6 <= hour < 12:
            return "Morning (06-12)"
        if 12 <= hour < 18:
            return "Afternoon (12-18)"
        if (18 <= hour <= 24) or (0 <= hour < 6):
            return "Evening/Night (18-06)"
    return "Unknown"


# Function for standardizing raw Fatal (Y/N) entries into Y, N, or Unknown
# Input: f_val (str or NaN) - raw fatality indicator
# Output: 'Y', 'N', or 'Unknown'
def clean_fatality(f_val):
    if not isinstance(f_val, str) or pd.isna(f_val):
        return "Unknown"
    f_clean = f_val.strip().upper()
    if f_clean in ("Y", "FATAL"):
        return "Y"
    if f_clean in ("N", "NOT FATAL"):
        return "N"
    return "Unknown"


# Function for cleaning and feature engineering the Global Shark Attack dataset
# Input: df (pd.DataFrame) - raw shark attacks DataFrame
# Output: cleaned DataFrame with Year, Month, Season, Country_clean, Area_clean, Activity_group, Species_group, Time_of_day, Fatal_clean
def clean_shark_data(df):
    # 1. Remove trailing completely empty rows present in GSAF CSV exports
    cleaned = df.dropna(how="all").copy()
    if "Case Number" in cleaned.columns:
        cleaned = cleaned[
            cleaned["Case Number"].notna()
            & (cleaned["Case Number"].astype(str).str.strip() != "")
            & (cleaned["Case Number"].astype(str).str.strip() != "0")
        ].copy()

    # 2. Drop administrative / duplicate export columns that carry no analytical value
    cols_to_drop = [
        "href", "href formula", "pdf", "Case Number.1", "Case Number.2",
        "original order", "Unnamed: 22", "Unnamed: 23", "Investigator or Source",
    ]
    cleaned = cleaned.drop(columns=[c for c in cols_to_drop if c in cleaned.columns])

    # 3. Filter reliable modern reporting window (1960 - 2023)
    cleaned["Year"] = pd.to_numeric(cleaned["Year"], errors="coerce")
    cleaned = cleaned[
        cleaned["Year"].notna() & (cleaned["Year"] >= 1960) & (cleaned["Year"] <= 2023)
    ].copy()
    cleaned["Year"] = cleaned["Year"].astype(int)

    # 4. Normalize Country names to match World Bank naming conventions
    raw_country = cleaned["Country"].astype(str).str.strip().str.rstrip("?").str.strip().str.upper()
    cleaned["Country_clean"] = raw_country.map(COUNTRY_MAPPING)
    cleaned["Country_clean"] = cleaned["Country_clean"].fillna(raw_country.str.title())
    cleaned["Country_clean"] = cleaned["Country_clean"].replace({"Nan": "Unknown", "": "Unknown"})

    # 5. Clean Area and Location text
    cleaned["Area_clean"] = (
        cleaned["Area"].astype(str).str.strip().str.title().replace({"Nan": "Unknown", "": "Unknown"})
    )
    cleaned["Location_clean"] = (
        cleaned["Location"].astype(str).str.strip().replace({"Nan": "Unknown", "": "Unknown"})
    )

    # 6. Extract Month and hemisphere-aware Season
    cleaned["Month"] = cleaned["Date"].apply(parse_month)
    cleaned["Season"] = cleaned.apply(lambda r: assign_season(r["Month"], r["Country_clean"]), axis=1)

    # 7. Categorize Activity, Species, Fatality, Time_of_day, Sex, Age, and Type
    cleaned["Activity_group"] = cleaned["Activity"].apply(categorize_activity)
    cleaned["Species_group"] = (
        cleaned["Species"].apply(categorize_species) if "Species" in cleaned.columns else "Unidentified"
    )
    cleaned["Fatal_clean"] = (
        cleaned["Fatal (Y/N)"].apply(clean_fatality) if "Fatal (Y/N)" in cleaned.columns else "Unknown"
    )
    cleaned["Time_of_day"] = (
        cleaned["Time"].apply(categorize_time) if "Time" in cleaned.columns else "Unknown"
    )
    if "Sex" in cleaned.columns:
        cleaned["Sex_clean"] = (
            cleaned["Sex"].astype(str).str.strip().str.upper().apply(lambda x: x if x in ("M", "F") else "Unknown")
        )
    else:
        cleaned["Sex_clean"] = "Unknown"
    cleaned["Age_clean"] = pd.to_numeric(cleaned["Age"], errors="coerce") if "Age" in cleaned.columns else np.nan
    cleaned["Type"] = cleaned["Type"].fillna("Unprovoked").astype(str).str.strip().str.title()

    return cleaned.reset_index(drop=True)


# Function for reshaping tourism data and merging it with annual shark attack counts
# Input: shark_df (pd.DataFrame) cleaned shark data, tourism_df (pd.DataFrame) raw World Bank data
# Output: merged DataFrame on (Country_clean, Year) covering 1995-2020 with annual attacks and arrivals
def clean_and_merge_tourism(shark_df, tourism_df):
    id_vars = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
    year_cols = [c for c in tourism_df.columns if c.isdigit()]

    # Reshape World Bank wide table into tidy long format
    tourism_long = pd.melt(
        tourism_df,
        id_vars=id_vars,
        value_vars=year_cols,
        var_name="Year",
        value_name="Arrivals",
    )
    tourism_long["Year"] = pd.to_numeric(tourism_long["Year"], errors="coerce")
    tourism_long["Arrivals"] = pd.to_numeric(tourism_long["Arrivals"], errors="coerce")
    tourism_long = tourism_long.dropna(subset=["Arrivals", "Year"]).copy()
    tourism_long["Year"] = tourism_long["Year"].astype(int)
    tourism_long = tourism_long[(tourism_long["Year"] >= 1995) & (tourism_long["Year"] <= 2020)].copy()
    tourism_long["Country_clean"] = tourism_long["Country Name"].astype(str).str.strip()

    # Aggregate shark attacks per Country and Year
    shark_agg = shark_df.groupby(["Country_clean", "Year"]).agg(
        annual_attacks=("Case Number", "count"),
        fatal_attacks=("Fatal_clean", lambda s: (s == "Y").sum()),
    ).reset_index()

    # Inner join on harmonized Country_clean and Year
    merged = pd.merge(
        tourism_long[["Country_clean", "Country Code", "Year", "Arrivals"]],
        shark_agg,
        on=["Country_clean", "Year"],
        how="inner",
    )
    return merged.sort_values(by=["Country_clean", "Year"]).reset_index(drop=True)

## Step 2: Query 1 — Grouping & Fatality Rate Analysis `[1 point]`

- **Calculation:** Groups cleaned incidents by `Country_clean`, `Activity_group`, and `Species_group`, computing `Total_Attacks`, `Fatal_Attacks`, and `Fatality_Rate_Pct = (Fatal_Attacks / Total_Attacks) * 100`.
- **What it represents:** Contrasts absolute attack frequency against lethality across nations, recreational activities, and identified shark species.
- **Interpretation:** The United States accounts for the largest share of attacks (1,756) with low lethality (~4.5%), whereas Australia and South Africa show much higher mortality. Surfing generates the most encounters (1,250) with low fatality (~4.5%), while Tiger (22.3%), White (18.0%), and Bull (16.0%) sharks drive nearly all identified fatal attacks.

In [3]:
# ==============================================================================
# Query 1: Grouping & Aggregation [1 point] — Owner: Baptiste Noailhac
# ==============================================================================

# Function for computing grouping indicators (top countries, activities, species, fatality rates)
# Input: shark_df (pd.DataFrame) cleaned shark data, top_n (int) number of top countries
# Output: dictionary with summary DataFrames and interactive Plotly bar charts
def query_grouping_analysis(shark_df, top_n=10):
    # 1. Country-level aggregation
    country_stats = shark_df.groupby("Country_clean").agg(
        Total_Attacks=("Case Number", "count"),
        Fatal_Attacks=("Fatal_clean", lambda s: (s == "Y").sum()),
        Non_Fatal_Attacks=("Fatal_clean", lambda s: (s == "N").sum()),
    ).reset_index()
    country_stats["Fatality_Rate_Pct"] = (
        (country_stats["Fatal_Attacks"] / country_stats["Total_Attacks"]) * 100
    ).round(2)
    top_countries = country_stats.sort_values(by="Total_Attacks", ascending=False).head(top_n).reset_index(drop=True)

    # 2. Activity-level aggregation
    activity_stats = shark_df.groupby("Activity_group").agg(
        Total_Attacks=("Case Number", "count"),
        Fatal_Attacks=("Fatal_clean", lambda s: (s == "Y").sum()),
    ).reset_index()
    activity_stats["Fatality_Rate_Pct"] = (
        (activity_stats["Fatal_Attacks"] / activity_stats["Total_Attacks"]) * 100
    ).round(2)
    activity_stats = activity_stats.sort_values(by="Total_Attacks", ascending=False).reset_index(drop=True)

    # 3. Species-level aggregation (focusing on identified species for fatality comparison)
    species_stats = shark_df.groupby("Species_group").agg(
        Total_Attacks=("Case Number", "count"),
        Fatal_Attacks=("Fatal_clean", lambda s: (s == "Y").sum()),
    ).reset_index()
    species_stats["Fatality_Rate_Pct"] = (
        (species_stats["Fatal_Attacks"] / species_stats["Total_Attacks"]) * 100
    ).round(2)
    species_stats = species_stats.sort_values(by="Total_Attacks", ascending=False).reset_index(drop=True)
    species_identified = species_stats[species_stats["Species_group"] != "Unidentified"].copy()

    # 4. Interactive Plotly Charts
    fig_countries = px.bar(
        top_countries,
        x="Total_Attacks",
        y="Country_clean",
        orientation="h",
        color="Fatality_Rate_Pct",
        color_continuous_scale="Tealgrn",
        labels={
            "Total_Attacks": "Number of Recorded Attacks",
            "Country_clean": "Country",
            "Fatality_Rate_Pct": "Fatality Rate (%)",
        },
        title=f"Top {top_n} Countries by Shark Attacks (1960-2018) & Fatality Rate",
    )
    fig_countries.update_layout(yaxis={"autorange": "reversed"}, template="plotly_white")

    fig_activities = px.bar(
        activity_stats,
        x="Activity_group",
        y="Total_Attacks",
        color="Fatality_Rate_Pct",
        color_continuous_scale="Reds",
        labels={
            "Total_Attacks": "Number of Attacks",
            "Activity_group": "Activity Category",
            "Fatality_Rate_Pct": "Fatality Rate (%)",
        },
        title="Shark Attacks by Activity Group & Associated Fatality Rate (%)",
    )
    fig_activities.update_layout(template="plotly_white")

    fig_species = px.bar(
        species_identified,
        x="Species_group",
        y="Total_Attacks",
        color="Fatality_Rate_Pct",
        color_continuous_scale="OrRd",
        labels={
            "Total_Attacks": "Identified Incidents",
            "Species_group": "Shark Species Group",
            "Fatality_Rate_Pct": "Fatality Rate (%)",
        },
        title="Identified Shark Species Groups: Incident Count & Fatality Rate (%)",
    )
    fig_species.update_layout(template="plotly_white")

    return {
        "top_countries": top_countries,
        "activity_stats": activity_stats,
        "species_stats": species_stats,
        "fig_countries": fig_countries,
        "fig_activities": fig_activities,
        "fig_species": fig_species,
    }


# ==============================================================================

## Step 3: Query 2 — Frequent Pattern Mining with Apriori `[2 points]`

- **Calculation:** Converts discrete incident features (`Country_clean`, `Activity_group`, `Type`, `Season`, `Time_of_day`, `Fatal_clean`) into a boolean transaction matrix, sweeps `min_support` from `0.15` to `0.02`, runs `apriori(min_support=0.03)`, and extracts `association_rules(metric='lift', min_threshold=1.05)` (Lab 9 methodology).
- **What it represents:** Discovers recurring multi-variable profiles associated with non-fatal vs. high-risk shark encounters.
- **Interpretation:** Itemset counts grow exponentially below `min_support = 0.05`. High-lift rules show strong co-occurrence between US summer afternoon surfing and non-fatal unprovoked bites, confirming distinct recreational encounter signatures.

In [4]:
# ==============================================================================
# Query 2: Frequent Pattern Mining (Apriori) [2 points] — Owner: Baptiste Noailhac
# ==============================================================================

# Function for extracting frequent itemsets and association rules using mlxtend Apriori
# Input: shark_df (pd.DataFrame), min_support (float), min_lift (float)
# Output: dictionary with support sensitivity, frequent itemsets, association rules, and Plotly figures
def query_frequent_patterns(shark_df, min_support=0.03, min_lift=1.05):
    cols = ["Country_clean", "Activity_group", "Type", "Season", "Time_of_day", "Fatal_clean"]
    df_cat = shark_df[cols].copy()

    # Exclude Unknown values so association rules reflect real observations
    for col in cols:
        df_cat[col] = df_cat[col].replace({"Unknown": np.nan})
    df_cat = df_cat.dropna()

    # Group minor countries into Other_Country to keep transaction space compact
    top_5_countries = shark_df["Country_clean"].value_counts().head(5).index.tolist()
    df_cat["Country_clean"] = df_cat["Country_clean"].apply(
        lambda c: c if c in top_5_countries else "Other_Country"
    )

    # Boolean one-hot transaction matrix (Lab 9 methodology)
    df_bool = pd.get_dummies(df_cat, dtype=bool)

    # 1. Support sensitivity sweep across thresholds
    support_thresholds = [0.15, 0.10, 0.08, 0.05, 0.04, 0.03, 0.02]
    sensitivity_counts = []
    for s in support_thresholds:
        itemsets_s = apriori(df_bool, min_support=s, use_colnames=True)
        sensitivity_counts.append(len(itemsets_s))
    sensitivity_df = pd.DataFrame({
        "min_support": support_thresholds,
        "pattern_count": sensitivity_counts,
    })

    fig_sensitivity = px.line(
        sensitivity_df,
        x="min_support",
        y="pattern_count",
        markers=True,
        labels={
            "min_support": "Minimal Support Threshold (min_support)",
            "pattern_count": "Number of Frequent Itemsets",
        },
        title="Apriori Support Sensitivity Curve (Lab 9 Pattern)",
    )
    fig_sensitivity.update_layout(template="plotly_white")

    # 2. Extract frequent itemsets and association rules
    frequent_itemsets = apriori(df_bool, min_support=min_support, use_colnames=True)
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=min_lift)
    rules["antecedents_str"] = rules["antecedents"].apply(lambda s: ", ".join(sorted(list(s))))
    rules["consequents_str"] = rules["consequents"].apply(lambda s: ", ".join(sorted(list(s))))
    rules_sorted = rules.sort_values(by=["lift", "confidence"], ascending=[False, False]).reset_index(drop=True)

    # 3. Interactive scatter plot of support vs. confidence colored by lift
    fig_rules = px.scatter(
        rules_sorted,
        x="support",
        y="confidence",
        color="lift",
        color_continuous_scale="Viridis",
        hover_data=["antecedents_str", "consequents_str"],
        labels={"support": "Support", "confidence": "Confidence", "lift": "Lift"},
        title="Association Rules: Support vs. Confidence (colored by Lift)",
    )
    fig_rules.update_layout(template="plotly_white")

    return {
        "n_transactions": int(len(df_bool)),
        "frequent_itemsets": frequent_itemsets,
        "rules": rules_sorted,
        "sensitivity_df": sensitivity_df,
        "fig_sensitivity": fig_sensitivity,
        "fig_rules": fig_rules,
    }


# ==============================================================================

## Step 4: Query 3 — Temporal Seasonality & Prophet Forecasting `[2 points]`

- **Calculation:** Compares normalized monthly distributions between Northern Hemisphere (USA) and Southern Hemisphere (Australia & South Africa), trains an annual `Prophet(interval_width=0.95)` model on complete years (1960-2017) for a 5-year forecast, and runs a monthly Prophet decomposition (`trend` + `yearly` seasonality, Lab 7 methodology).
- **What it represents:** Demonstrates seasonal coupling with local summer beach attendance and projects expected annual encounter baselines with 95% confidence bounds.
- **Interpretation:** Hemispheres display mirror-image seasonality (July-September in the US vs. December-March in Australia/South Africa). Annual global counts are forecasted to remain around 112-118 incidents/year over the 5-year horizon.

In [5]:
# ==============================================================================
# Query 3: Temporal Analysis & Forecasting (Prophet) [2 points] — Owner: Romain Thepaut
# ==============================================================================

# Function for analyzing hemisphere seasonality and forecasting shark attacks with Prophet
# Input: shark_df (pd.DataFrame), forecast_years (int)
# Output: dictionary with seasonality table, Prophet forecasts, components, and Plotly figures
def query_temporal_forecast(shark_df, forecast_years=5):
    # 1. Northern vs. Southern Hemisphere monthly seasonality
    df_valid_months = shark_df[shark_df["Month"].notna() & shark_df["Month"].isin(range(1, 13))].copy()
    df_valid_months["Month"] = df_valid_months["Month"].astype(int)

    us_months = (
        df_valid_months[df_valid_months["Country_clean"] == "United States"]["Month"]
        .value_counts(normalize=True)
        .sort_index() * 100
    )
    south_months = (
        df_valid_months[df_valid_months["Country_clean"].isin(["Australia", "South Africa"])]["Month"]
        .value_counts(normalize=True)
        .sort_index() * 100
    )

    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    season_df = pd.DataFrame({
        "Month": month_names,
        "Northern_Hemisphere_US": [round(float(us_months.get(m, 0.0)), 2) for m in range(1, 13)],
        "Southern_Hemisphere_Aus_SA": [round(float(south_months.get(m, 0.0)), 2) for m in range(1, 13)],
    })

    fig_seasonality = go.Figure()
    fig_seasonality.add_trace(go.Scatter(
        x=season_df["Month"],
        y=season_df["Northern_Hemisphere_US"],
        mode="lines+markers",
        name="United States (Northern Hemisphere)",
        line=dict(color="#0284C7", width=3),
    ))
    fig_seasonality.add_trace(go.Scatter(
        x=season_df["Month"],
        y=season_df["Southern_Hemisphere_Aus_SA"],
        mode="lines+markers",
        name="Australia & South Africa (Southern Hemisphere)",
        line=dict(color="#DC2626", width=3),
    ))
    fig_seasonality.update_layout(
        title="Monthly Seasonality Comparison: Northern vs. Southern Hemisphere (% of Incidents)",
        xaxis_title="Month",
        yaxis_title="Share of Incidents (%)",
        template="plotly_white",
    )

    # 2. Annual Prophet 5-Year Forecast (excluding incomplete final partial year if < 60% of prior mean)
    annual_counts = shark_df.groupby("Year").size().reset_index(name="y").sort_values("Year")
    recent_avg = annual_counts["y"].iloc[-6:-1].mean()
    if len(annual_counts) > 5 and annual_counts["y"].iloc[-1] < 0.6 * recent_avg:
        annual_counts = annual_counts.iloc[:-1].copy()

    annual_counts["ds"] = pd.to_datetime(annual_counts["Year"].astype(str) + "-01-01")
    df_prophet_annual = annual_counts[["ds", "y"]].reset_index(drop=True)

    model_annual = Prophet(
        interval_width=0.95,
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False,
    )
    model_annual.fit(df_prophet_annual)
    future_annual = model_annual.make_future_dataframe(periods=forecast_years, freq="YS")
    forecast_annual = model_annual.predict(future_annual)

    fig_forecast = go.Figure()
    fig_forecast.add_trace(go.Scatter(
        x=df_prophet_annual["ds"],
        y=df_prophet_annual["y"],
        mode="markers",
        name="Observed Annual Incidents",
        marker=dict(color="#0F172A", size=6),
    ))
    fig_forecast.add_trace(go.Scatter(
        x=forecast_annual["ds"],
        y=forecast_annual["yhat"],
        mode="lines",
        name="Prophet Forecast (yhat)",
        line=dict(color="#0F766E", width=3),
    ))
    fig_forecast.add_trace(go.Scatter(
        x=forecast_annual["ds"],
        y=forecast_annual["yhat_upper"],
        mode="lines",
        name="95% Upper Bound",
        line=dict(color="#94A3B8", width=1, dash="dash"),
    ))
    fig_forecast.add_trace(go.Scatter(
        x=forecast_annual["ds"],
        y=forecast_annual["yhat_lower"],
        mode="lines",
        name="95% Lower Bound",
        line=dict(color="#94A3B8", width=1, dash="dash"),
        fill="tonexty",
        fillcolor="rgba(148, 163, 184, 0.2)",
    ))
    fig_forecast.update_layout(
        title=f"Global Shark Attacks Annual Trend & {forecast_years}-Year Prophet Forecast (95% CI)",
        xaxis_title="Year",
        yaxis_title="Annual Incident Count",
        template="plotly_white",
    )

    # 3. Monthly Prophet Decomposition (Trend + Yearly Seasonality Component, Lab 7 style)
    max_complete_year = int(annual_counts["Year"].max())
    monthly_df = df_valid_months[df_valid_months["Year"] <= max_complete_year].copy()
    monthly_df["ds"] = pd.to_datetime(
        monthly_df["Year"].astype(str) + "-" + monthly_df["Month"].astype(str).str.zfill(2) + "-01"
    )
    monthly_counts = monthly_df.groupby("ds").size().reset_index(name="y").sort_values("ds")

    model_monthly = Prophet(
        interval_width=0.95,
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
    )
    model_monthly.fit(monthly_counts)
    future_monthly = model_monthly.make_future_dataframe(periods=forecast_years * 12, freq="MS")
    forecast_monthly = model_monthly.predict(future_monthly)

    # Extract clean trend & yearly seasonality subplot
    one_year = forecast_monthly.tail(12).copy()
    one_year["Month_Num"] = one_year["ds"].dt.month
    one_year = one_year.sort_values("Month_Num")
    one_year["Month_Name"] = [month_names[m - 1] for m in one_year["Month_Num"]]

    fig_components = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Long-Term Monthly Trend Component", "Yearly Seasonal Effect (Monthly)"),
    )
    fig_components.add_trace(
        go.Scatter(x=forecast_monthly["ds"], y=forecast_monthly["trend"], mode="lines", name="Trend", line=dict(color="#0F766E", width=2.5)),
        row=1, col=1,
    )
    fig_components.add_trace(
        go.Bar(x=one_year["Month_Name"], y=one_year["yearly"], name="Yearly Seasonality", marker_color="#0284C7"),
        row=1, col=2,
    )
    fig_components.update_layout(
        title="Prophet Monthly Time-Series Decomposition (Trend & Seasonality Components)",
        template="plotly_white",
        showlegend=False,
    )

    return {
        "season_df": season_df,
        "fig_seasonality": fig_seasonality,
        "forecast_df": forecast_annual[["ds", "yhat", "yhat_lower", "yhat_upper"]],
        "monthly_forecast_df": forecast_monthly[["ds", "yhat", "trend", "yearly"]],
        "fig_forecast": fig_forecast,
        "fig_components": fig_components,
        "prophet_model": model_annual,
    }


# ==============================================================================

## Step 5: Query 4 — Spatial Analysis & Coastal Hotspot Clustering `[2 points]`

- **Calculation:** Geocodes `(Country_clean, Area_clean)` pairs (`>95%` coverage), builds a `GeoDataFrame` (`EPSG:4326`), performs `gpd.sjoin` against Natural Earth country polygons (`ne_110m_admin_0_countries.shp`), and applies **K-Means** ($k=6$, dynamically labeled from actual cluster centroids) alongside **OPTICS** density clustering (Lab 8 methodology).
- **What it represents:** Partitions global shark attacks into major oceanic coastal basins and contrasts localized fatality profiles.
- **Interpretation:** K-Means cleanly isolates 6 oceanic epicenters (US Atlantic/Florida, Australia/SW Pacific, Southern Africa/Indian Ocean, US West Coast/Hawaii, South America/Brazil, and Europe/Mediterranean), revealing stark mortality differences between low-lethality surfing hubs (US East Coast ~5%) and high-lethality basins (Australia ~18%, Southern Africa ~17%).

In [6]:
# ==============================================================================
# Query 4: Spatial Analysis & Clustering (GeoPandas + OPTICS/KMeans) [2 points] — Owner: Louis Le Forestier
# ==============================================================================

# Accurate coastal area coordinates for major GSAF administrative zones
COASTAL_AREA_COORDINATES = {
    ("United States", "Florida"): (28.0836, -80.6081),
    ("United States", "Hawaii"): (20.7984, -156.3319),
    ("United States", "California"): (34.4208, -119.6982),
    ("United States", "South Carolina"): (32.7765, -79.9311),
    ("United States", "North Carolina"): (34.2257, -77.9447),
    ("United States", "Texas"): (27.8006, -97.3964),
    ("United States", "New Jersey"): (39.3643, -74.4229),
    ("United States", "Oregon"): (44.6368, -124.0535),
    ("United States", "New York"): (40.5884, -73.6579),
    ("United States", "Virginia"): (36.8529, -75.9780),
    ("United States", "Georgia"): (31.1499, -81.4915),
    ("United States", "Alabama"): (30.2460, -87.7008),
    ("United States", "Massachusetts"): (41.6688, -70.2962),
    ("United States", "Louisiana"): (29.2500, -90.0000),
    ("United States", "Puerto Rico"): (18.4655, -66.1057),
    ("Australia", "New South Wales"): (-33.8688, 151.2093),
    ("Australia", "Queensland"): (-27.4698, 153.0251),
    ("Australia", "Western Australia"): (-31.9505, 115.8605),
    ("Australia", "South Australia"): (-34.9285, 138.6007),
    ("Australia", "Victoria"): (-38.3300, 144.3200),
    ("Australia", "Tasmania"): (-42.8821, 147.3272),
    ("Australia", "Northern Territory"): (-12.4634, 130.8456),
    ("Australia", "Torres Strait"): (-10.5833, 142.2167),
    ("South Africa", "Kwazulu-Natal"): (-29.8587, 31.0218),
    ("South Africa", "Eastern Cape Province"): (-33.9608, 25.6022),
    ("South Africa", "Western Cape Province"): (-34.1167, 18.4667),
    ("Brazil", "Pernambuco"): (-8.0578, -34.8829),
    ("Brazil", "Rio De Janeiro"): (-22.9068, -43.1729),
    ("Brazil", "Bahia"): (-12.9714, -38.5014),
    ("Brazil", "Maranhao"): (-2.5307, -44.3068),
    ("Brazil", "Santa Catarina"): (-27.5954, -48.5480),
    ("Bahamas, The", "Abaco Islands"): (26.5412, -77.0636),
    ("Bahamas, The", "Grand Bahama"): (26.6594, -78.4975),
    ("Bahamas, The", "New Providence"): (25.0443, -77.3504),
    ("New Zealand", "North Island"): (-36.8485, 174.7633),
    ("New Zealand", "South Island"): (-43.5321, 172.6362),
    ("Mexico", "Quintana Roo"): (20.6296, -87.0739),
    ("Mexico", "Guerrero"): (16.8531, -99.8237),
    ("Mexico", "Baja California"): (24.1426, -110.3128),
    ("Mexico", "Veracruz"): (19.1738, -96.1342),
    ("Reunion", "Saint-Paul"): (-21.0097, 55.2707),
    ("Reunion", "Saint-Gilles"): (-21.0543, 55.2255),
    ("Reunion", "Saint-Leu"): (-21.1706, 55.2882),
}

# Country-level coastal fallback centroids to ensure >90% global geocoding coverage
COUNTRY_COASTAL_CENTROIDS = {
    "United States": (28.50, -81.00),
    "Australia": (-28.00, 152.50),
    "South Africa": (-32.50, 28.00),
    "Brazil": (-10.00, -35.50),
    "Bahamas, The": (25.03, -77.40),
    "New Zealand": (-37.78, 175.28),
    "Mexico": (20.50, -96.50),
    "Reunion": (-21.11, 55.53),
    "Papua New Guinea": (-6.31, 147.00),
    "Fiji": (-17.71, 178.07),
    "New Caledonia": (-21.25, 165.30),
    "French Polynesia": (-17.65, -149.43),
    "Egypt, Arab Rep.": (27.26, 33.81),
    "Italy": (40.85, 14.27),
    "Spain": (39.47, -0.38),
    "France": (43.30, 5.37),
    "Japan": (33.59, 130.40),
    "Philippines": (12.88, 121.77),
    "Indonesia": (-8.41, 115.19),
    "Mozambique": (-23.86, 35.38),
    "Cuba": (23.11, -82.37),
    "Costa Rica": (9.75, -84.62),
    "Ecuador": (-1.83, -80.75),
    "Chile": (-33.05, -71.62),
    "United Kingdom": (50.37, -4.14),
    "Hong Kong SAR, China": (22.32, 114.17),
    "Venezuela, RB": (10.60, -66.93),
    "Colombia": (10.39, -75.48),
    "Jamaica": (18.11, -77.30),
    "Bermuda": (32.31, -64.75),
    "Taiwan": (23.70, 121.50),
    "India": (15.30, 73.92),
    "Thailand": (7.88, 98.39),
    "Vietnam": (12.24, 109.19),
    "Viet Nam": (12.24, 109.19),
    "Turkiye": (36.88, 30.70),
    "Greece": (37.98, 23.73),
    "Croatia": (43.51, 16.44),
    "Canada": (44.65, -63.57),
    "Solomon Islands": (-9.43, 159.96),
    "Tonga": (-21.18, -175.20),
    "Samoa": (-13.83, -171.76),
    "Vanuatu": (-17.73, 168.32),
    "Seychelles": (-4.68, 55.49),
    "Mauritius": (-20.35, 57.55),
    "Kenya": (-4.04, 39.67),
    "Madagascar": (-18.77, 49.40),
    "Panama": (8.98, -79.52),
    "Dominican Republic": (18.49, -69.93),
    "Belize": (17.50, -88.20),
    "Honduras": (16.33, -86.53),
    "Nicaragua": (12.11, -86.24),
    "Israel": (32.09, 34.78),
    "Iran, Islamic Rep.": (27.18, 56.28),
    "Korea, Rep.": (35.18, 129.08),
    "Russian Federation": (43.12, 131.89),
    "China": (22.54, 114.06),
}


# Function for geocoding unique coastal (Country_clean, Area_clean) pairs and saving local cache
# Input: shark_df (pd.DataFrame), cache_path (str or None), force_refresh (bool)
# Output: DataFrame with columns ['Country_clean', 'Area_clean', 'latitude', 'longitude', 'geo_precision']
def geocode_coastal_areas(shark_df, cache_path=None, force_refresh=False):
    if cache_path and os.path.exists(cache_path) and not force_refresh:
        cached = pd.read_csv(cache_path)
        if "geo_precision" in cached.columns and len(cached) > 100:
            return cached

    unique_pairs = shark_df[["Country_clean", "Area_clean"]].drop_duplicates().copy()
    geo_rows = []
    for _, row in unique_pairs.iterrows():
        country = row["Country_clean"]
        area = row["Area_clean"]
        if (country, area) in COASTAL_AREA_COORDINATES:
            lat, lon = COASTAL_AREA_COORDINATES[(country, area)]
            geo_rows.append({
                "Country_clean": country,
                "Area_clean": area,
                "latitude": lat,
                "longitude": lon,
                "geo_precision": "area",
            })
        elif country in COUNTRY_COASTAL_CENTROIDS:
            lat, lon = COUNTRY_COASTAL_CENTROIDS[country]
            geo_rows.append({
                "Country_clean": country,
                "Area_clean": area,
                "latitude": lat,
                "longitude": lon,
                "geo_precision": "country_centroid",
            })

    geo_df = pd.DataFrame(geo_rows)
    if cache_path:
        dir_name = os.path.dirname(cache_path)
        if dir_name:
            os.makedirs(dir_name, exist_ok=True)
        geo_df.to_csv(cache_path, index=False)

    return geo_df


# Function for dynamically naming a spatial cluster based on its actual geographic centroid and top countries
# Input: cluster_subset (pd.DataFrame) subset of incidents belonging to one cluster
# Output: human-readable geographic basin name (str)
def label_spatial_cluster(cluster_subset):
    mean_lat = float(cluster_subset["latitude"].mean())
    mean_lon = float(cluster_subset["longitude"].mean())
    top_countries = cluster_subset["Country_clean"].value_counts().head(2).index.tolist()
    top_str = " / ".join(top_countries)

    # Geographic basin heuristics derived strictly from actual centroid coordinates
    if mean_lon < -110:
        return f"Pacific & US West Coast / Hawaii ({top_str})"
    if -110 <= mean_lon <= -30:
        if mean_lat >= 20:
            return f"US Atlantic, Florida & Caribbean ({top_str})"
        return f"South America & Tropical Americas ({top_str})"
    if -30 < mean_lon <= 65:
        if mean_lat < 0:
            return f"Southern Africa & Indian Ocean ({top_str})"
        return f"Europe, Mediterranean & Red Sea ({top_str})"
    return f"Australia & South-West Pacific ({top_str})"


# Function for running GeoPandas spatial join and K-Means / OPTICS spatial clustering (Lab 8 pattern)
# Input: shark_df (pd.DataFrame), geo_df (pd.DataFrame), shp_path (str or None), n_clusters (int)
# Output: dictionary with clustered GeoDataFrame, K-Means & OPTICS summaries, and Plotly world maps
def query_spatial_clustering(shark_df, geo_df, shp_path=None, n_clusters=6):
    merged_geo = pd.merge(shark_df, geo_df, on=["Country_clean", "Area_clean"], how="inner").copy()
    coverage_pct = round((len(merged_geo) / len(shark_df)) * 100, 2) if len(shark_df) > 0 else 0.0

    # 1. Build GeoDataFrame (Lab 8 pattern)
    geometry = [Point(lon, lat) for lon, lat in zip(merged_geo["longitude"], merged_geo["latitude"])]
    gdf = gpd.GeoDataFrame(merged_geo, geometry=geometry, crs="EPSG:4326")

    # 2. Spatial join with Natural Earth countries shapefile when available
    sjoin_matched = 0
    if shp_path and os.path.exists(shp_path):
        world_raw = gpd.read_file(shp_path).to_crs("EPSG:4326")
        admin_col = "admin" if "admin" in world_raw.columns else ("ADMIN" if "ADMIN" in world_raw.columns else None)
        cont_col = "continent" if "continent" in world_raw.columns else ("CONTINENT" if "CONTINENT" in world_raw.columns else None)
        keep_cols = [c for c in [admin_col, cont_col, "geometry"] if c is not None]
        world = world_raw[keep_cols].copy()
        if admin_col:
            world = world.rename(columns={admin_col: "shp_admin"})
        if cont_col:
            world = world.rename(columns={cont_col: "shp_continent"})
        gdf = gpd.sjoin(gdf, world, how="left", predicate="intersects")
        gdf = gdf.drop(columns=["index_right"], errors="ignore")
        if "shp_admin" in gdf.columns:
            sjoin_matched = int(gdf["shp_admin"].notna().sum())

    coords = gdf[["latitude", "longitude"]].values

    # 3. K-Means spatial clustering with dynamic centroid-based cluster labeling
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    gdf["kmeans_cluster"] = kmeans.fit_predict(coords)

    dynamic_labels = {}
    for cid in sorted(gdf["kmeans_cluster"].unique()):
        subset = gdf[gdf["kmeans_cluster"] == cid]
        dynamic_labels[cid] = label_spatial_cluster(subset)
    gdf["Cluster_Name"] = gdf["kmeans_cluster"].map(dynamic_labels)

    # 4. OPTICS density-based spatial clustering (Lab 8 comparison)
    optics = OPTICS(min_samples=25, xi=0.05, min_cluster_size=0.04)
    gdf["optics_cluster"] = optics.fit_predict(coords)
    gdf["OPTICS_Label"] = gdf["optics_cluster"].apply(
        lambda c: "Noise / Isolated Coast (-1)" if c == -1 else f"Dense Hotspot #{c}"
    )

    # 5. Summarize K-Means clusters
    cluster_summary = gdf.groupby("Cluster_Name").agg(
        Total_Incidents=("Case Number", "count"),
        Fatal_Count=("Fatal_clean", lambda s: (s == "Y").sum()),
        Mean_Lat=("latitude", "mean"),
        Mean_Lon=("longitude", "mean"),
        Dominant_Activity=("Activity_group", lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else "Unknown"),
    ).reset_index()
    cluster_summary["Fatality_Rate_Pct"] = (
        (cluster_summary["Fatal_Count"] / cluster_summary["Total_Incidents"]) * 100
    ).round(2)
    cluster_summary["Mean_Lat"] = cluster_summary["Mean_Lat"].round(2)
    cluster_summary["Mean_Lon"] = cluster_summary["Mean_Lon"].round(2)
    cluster_summary = cluster_summary.sort_values(by="Total_Incidents", ascending=False).reset_index(drop=True)

    # Summarize OPTICS clusters
    optics_summary = gdf.groupby("OPTICS_Label").agg(
        Total_Incidents=("Case Number", "count"),
        Top_Country=("Country_clean", lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else "Unknown"),
    ).reset_index().sort_values(by="Total_Incidents", ascending=False).reset_index(drop=True)

    # 6. Aggregate by coastal point for clean, fast Plotly world maps (bubble size = incident count)
    point_agg = gdf.groupby(["Country_clean", "Area_clean", "latitude", "longitude", "Cluster_Name"]).agg(
        Incidents=("Case Number", "count"),
        Fatal_Count=("Fatal_clean", lambda s: (s == "Y").sum()),
        Top_Activity=("Activity_group", lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else "Unknown"),
    ).reset_index()
    point_agg["Fatality_Rate_Pct"] = ((point_agg["Fatal_Count"] / point_agg["Incidents"]) * 100).round(2)

    fig_map = px.scatter_geo(
        point_agg,
        lat="latitude",
        lon="longitude",
        size="Incidents",
        size_max=32,
        color="Cluster_Name",
        hover_name="Area_clean",
        hover_data={
            "Country_clean": True,
            "Incidents": True,
            "Fatality_Rate_Pct": True,
            "Top_Activity": True,
            "latitude": False,
            "longitude": False,
        },
        title=f"Global Shark Attack Coastal Hotspots — K-Means Clusters (k={n_clusters}, {coverage_pct}% coverage)",
        projection="natural earth",
    )
    fig_map.update_geos(showcoastlines=True, coastlinecolor="#94A3B8", showland=True, landcolor="#F1F5F9")
    fig_map.update_layout(template="plotly_white", legend=dict(orientation="h", yanchor="bottom", y=-0.25))

    optics_points = gdf.groupby(["Country_clean", "Area_clean", "latitude", "longitude", "OPTICS_Label"]).agg(
        Incidents=("Case Number", "count"),
    ).reset_index()
    fig_optics_map = px.scatter_geo(
        optics_points,
        lat="latitude",
        lon="longitude",
        size="Incidents",
        size_max=28,
        color="OPTICS_Label",
        hover_name="Area_clean",
        hover_data={"Country_clean": True, "Incidents": True, "latitude": False, "longitude": False},
        title="Density-Based Coastal Hotspots — OPTICS Clustering (Lab 8 Comparison)",
        projection="natural earth",
    )
    fig_optics_map.update_geos(showcoastlines=True, coastlinecolor="#94A3B8", showland=True, landcolor="#F1F5F9")
    fig_optics_map.update_layout(template="plotly_white", legend=dict(orientation="h", yanchor="bottom", y=-0.25))

    return {
        "geo_df": gdf,
        "coverage_pct": coverage_pct,
        "sjoin_matched": sjoin_matched,
        "cluster_summary": cluster_summary,
        "optics_summary": optics_summary,
        "fig_map": fig_map,
        "fig_optics_map": fig_optics_map,
        "kmeans_model": kmeans,
        "optics_model": optics,
    }


# ==============================================================================

## Step 6: External Dataset Bonus — World Bank International Tourism `[+1 Extra Point]`

- **Calculation:** Inner-joins annual shark attacks with World Bank inbound tourist arrivals (`ST.INT.ARVL`) on `(Country_clean, Year)` across **1995-2018**, computes `attacks_per_1M_tourists = (annual_attacks / Arrivals) * 1_000_000`, and evaluates Pearson ($r$) and Spearman ($\rho$) correlations.
- **What it represents:** Distinguishes sheer volume driven by mass tourism from per-visitor encounter risk.
- **Interpretation:** A strong positive correlation ($r = 0.5967, p < 0.001$) confirms that rising raw attack totals track growing coastal tourism. Normalizing per 1M tourists reverses the ranking: the United States drops to **0.39 attacks/1M tourists**, while **Australia (3.14)** is ~8x higher and **New Caledonia (18.31)** exhibits the highest relative risk.

In [7]:
# ==============================================================================
# External Bonus Query: World Bank Tourism Correlation [+1 point] — Owner: Romain Thepaut
# ==============================================================================

# Function for computing tourism-adjusted shark attack rates and correlations (1995-2020)
# Input: merged_df (pd.DataFrame) from clean_and_merge_tourism()
# Output: dictionary with enriched DataFrame, correlations, country ranking, and Plotly figures
def query_tourism_enrichment(merged_df):
    df_bonus = merged_df.copy()

    # 1. Compute Shark Attacks per 1 Million International Tourists
    df_bonus["attacks_per_1M_tourists"] = (
        (df_bonus["annual_attacks"] / df_bonus["Arrivals"]) * 1_000_000
    ).round(4)

    # 2. Global Pearson and Spearman correlation across country-year observations
    p_corr, p_val = pearsonr(df_bonus["Arrivals"], df_bonus["annual_attacks"])
    s_corr, s_val = spearmanr(df_bonus["Arrivals"], df_bonus["annual_attacks"])

    # 3. Country-level comparison for Top 8 coastal countries by attack volume (1995-2018)
    country_comparison = df_bonus.groupby("Country_clean").agg(
        Years_Observed=("Year", "nunique"),
        Total_Attacks=("annual_attacks", "sum"),
        Total_Tourist_Arrivals=("Arrivals", "sum"),
    ).reset_index()
    country_comparison["attacks_per_1M_tourists"] = (
        (country_comparison["Total_Attacks"] / country_comparison["Total_Tourist_Arrivals"]) * 1_000_000
    ).round(3)
    country_comparison = (
        country_comparison.sort_values(by="Total_Attacks", ascending=False).head(8).reset_index(drop=True)
    )

    # 4. Clean Two-Panel Comparison Chart (no arbitrary visual multiplier)
    by_risk = country_comparison.sort_values(by="attacks_per_1M_tourists", ascending=False)
    fig_comparison = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "Raw Volume: Total Shark Attacks (1995-2018)",
            "Tourism-Normalized Risk: Attacks per 1M Tourists",
        ),
    )
    fig_comparison.add_trace(
        go.Bar(
            x=country_comparison["Country_clean"],
            y=country_comparison["Total_Attacks"],
            name="Total Attacks",
            marker_color="#0284C7",
            text=country_comparison["Total_Attacks"],
            textposition="outside",
        ),
        row=1, col=1,
    )
    fig_comparison.add_trace(
        go.Bar(
            x=by_risk["Country_clean"],
            y=by_risk["attacks_per_1M_tourists"],
            name="Attacks / 1M Tourists",
            marker_color="#DC2626",
            text=by_risk["attacks_per_1M_tourists"],
            textposition="outside",
        ),
        row=1, col=2,
    )
    fig_comparison.update_layout(
        title="Raw Incident Volume vs. Tourism-Adjusted Risk Rate (World Bank ST.INT.ARVL)",
        template="plotly_white",
        showlegend=False,
    )

    # 5. Dual-Axis Time Series for United States (1995-2018)
    us_data = df_bonus[df_bonus["Country_clean"] == "United States"].sort_values("Year")
    fig_dual_axis = go.Figure()
    fig_dual_axis.add_trace(go.Scatter(
        x=us_data["Year"],
        y=us_data["annual_attacks"],
        name="US Annual Shark Attacks",
        mode="lines+markers",
        line=dict(color="#0F172A", width=3),
    ))
    fig_dual_axis.add_trace(go.Scatter(
        x=us_data["Year"],
        y=us_data["Arrivals"] / 1_000_000,
        name="US Tourist Arrivals (Millions)",
        mode="lines+markers",
        line=dict(color="#0F766E", width=3),
        yaxis="y2",
    ))
    fig_dual_axis.update_layout(
        title="United States: Annual Shark Attacks vs. Inbound Tourist Arrivals (1995-2018)",
        xaxis=dict(title="Year"),
        yaxis=dict(title=dict(text="Annual Attacks", font=dict(color="#0F172A"))),
        yaxis2=dict(
            title=dict(text="Tourist Arrivals (M)", font=dict(color="#0F766E")),
            overlaying="y",
            side="right",
        ),
        template="plotly_white",
    )

    return {
        "enriched_df": df_bonus,
        "pearson_corr": float(p_corr),
        "pearson_pval": float(p_val),
        "spearman_corr": float(s_corr),
        "spearman_pval": float(s_val),
        "country_comparison": country_comparison,
        "fig_comparison": fig_comparison,
        "fig_dual_axis": fig_dual_axis,
    }

## Step 7: Interactive Dash Dashboard & Offline HTML Proof `[2 points]`

The functions below compute dynamic KPI cards from the live DataFrames, assemble a 5-tab **Python Dash** application with interactive filters (`@app.callback`) and mandatory team/dataset metadata, and export a standalone offline proof file (`Laurent_dashboard.html`).

In [8]:
# Project metadata (mandatory on the dashboard per Final_project.pdf)
PROJECT_TITLE = "Global Shark Attacks Analysis & International Tourism Risk"
DATASET_NAMES = (
    "Primary: Global Shark Attack (GSAF - Kaggle / sharkattackfile.net) | "
    "External Bonus: International Tourism Arrivals (World Bank ST.INT.ARVL)"
)
TEAM_MEMBERS = [
    "Maxime Laurent (Team Leader)",
    "Louis Le Forestier",
    "Baptiste Noailhac",
    "Romain Thepaut",
]


# Function for dynamically computing headline KPI metrics from cleaned data and bonus results
# Input: clean_sharks (pd.DataFrame), bonus_results (dict)
# Output: dictionary of formatted KPI values and subtitles
def compute_dashboard_kpis(clean_sharks, bonus_results):
    total_incidents = int(len(clean_sharks))
    fatal_count = int((clean_sharks["Fatal_clean"] == "Y").sum())
    fatality_rate = round((fatal_count / total_incidents) * 100, 1) if total_incidents > 0 else 0.0
    min_year = int(clean_sharks["Year"].min())
    max_year = int(clean_sharks["Year"].max())

    top_country = clean_sharks["Country_clean"].value_counts().index[0]
    top_country_count = int(clean_sharks["Country_clean"].value_counts().iloc[0])
    top_country_share = round((top_country_count / total_incidents) * 100, 1)

    comp = bonus_results["country_comparison"].sort_values("attacks_per_1M_tourists", ascending=False)
    highest_risk_country = str(comp.iloc[0]["Country_clean"])
    highest_risk_rate = float(comp.iloc[0]["attacks_per_1M_tourists"])
    us_rows = comp[comp["Country_clean"] == "United States"]
    us_rate = float(us_rows.iloc[0]["attacks_per_1M_tourists"]) if len(us_rows) > 0 else 0.0

    return {
        "total_incidents": f"{total_incidents:,}",
        "year_window": f"Modern window: {min_year} - {max_year}",
        "fatality_rate": f"{fatality_rate} %",
        "fatal_subtitle": f"{fatal_count:,} fatal outcomes documented",
        "top_country": top_country,
        "top_country_subtitle": f"{top_country_count:,} attacks ({top_country_share} % of total)",
        "highest_risk_label": f"{highest_risk_country} ({highest_risk_rate:.2f})",
        "highest_risk_subtitle": f"Attacks / 1M tourists (vs US {us_rate:.2f})",
        "pearson_r": round(float(bonus_results["pearson_corr"]), 4),
    }


# Function for building the interactive Python Dash application with dynamic KPIs and filter callbacks
# Input: figures_dict (dict), kpis (dict), clean_sharks (pd.DataFrame or None), geo_df (pd.DataFrame or None)
# Output: configured Dash app instance
def create_dash_app(figures_dict=None, kpis=None, clean_sharks=None, geo_df=None):
    if figures_dict is None:
        figures_dict = {}
    if kpis is None:
        kpis = {
            "total_incidents": "N/A",
            "year_window": "Load dataset",
            "fatality_rate": "N/A",
            "fatal_subtitle": "",
            "top_country": "N/A",
            "top_country_subtitle": "",
            "highest_risk_label": "N/A",
            "highest_risk_subtitle": "",
            "pearson_r": 0.0,
        }

    app = dash.Dash(
        __name__,
        external_stylesheets=[dbc.themes.FLATLY],
        title="Global Shark Attacks KDD Dashboard - ESILV",
    )

    # Filter options derived dynamically from clean_sharks if provided
    if clean_sharks is not None and len(clean_sharks) > 0:
        min_yr = int(clean_sharks["Year"].min())
        max_yr = int(clean_sharks["Year"].max())
        top_countries = clean_sharks["Country_clean"].value_counts().head(15).index.tolist()
        activities = sorted(clean_sharks["Activity_group"].dropna().unique().tolist())
    else:
        min_yr, max_yr = 1960, 2018
        top_countries = ["United States", "Australia", "South Africa"]
        activities = ["Surfing", "Swimming", "Diving/Snorkeling"]

    # Mandatory Header Card (Dataset names + all 4 team members)
    header = dbc.Row(
        dbc.Col(
            html.Div(
                [
                    html.H2(PROJECT_TITLE, className="text-dark fw-bold mb-2"),
                    html.P(DATASET_NAMES, className="text-muted mb-2"),
                    html.Div(
                        [
                            html.Span("Team Members: ", className="fw-bold text-secondary me-2"),
                            html.Span(" | ".join(TEAM_MEMBERS), className="badge bg-secondary fs-6 text-wrap"),
                        ]
                    ),
                ],
                className="p-4 my-3 bg-light border rounded shadow-sm",
            )
        )
    )

    # Dynamic KPI summary cards
    kpi_cards = dbc.Row(
        [
            dbc.Col(
                dbc.Card(
                    dbc.CardBody(
                        [
                            html.H6("Total Analyzed Incidents", className="text-muted"),
                            html.H3(kpis["total_incidents"], className="text-primary fw-bold"),
                            html.Small(kpis["year_window"], className="text-muted"),
                        ]
                    ),
                    className="shadow-sm mb-3",
                ),
                width=3,
            ),
            dbc.Col(
                dbc.Card(
                    dbc.CardBody(
                        [
                            html.H6("Overall Fatality Rate", className="text-muted"),
                            html.H3(kpis["fatality_rate"], className="text-danger fw-bold"),
                            html.Small(kpis["fatal_subtitle"], className="text-muted"),
                        ]
                    ),
                    className="shadow-sm mb-3",
                ),
                width=3,
            ),
            dbc.Col(
                dbc.Card(
                    dbc.CardBody(
                        [
                            html.H6("Top Incident Country", className="text-muted"),
                            html.H3(kpis["top_country"], className="text-success fw-bold"),
                            html.Small(kpis["top_country_subtitle"], className="text-muted"),
                        ]
                    ),
                    className="shadow-sm mb-3",
                ),
                width=3,
            ),
            dbc.Col(
                dbc.Card(
                    dbc.CardBody(
                        [
                            html.H6("Highest Tourism Risk Rate", className="text-muted"),
                            html.H3(kpis["highest_risk_label"], className="text-warning fw-bold"),
                            html.Small(kpis["highest_risk_subtitle"], className="text-muted"),
                        ]
                    ),
                    className="shadow-sm mb-3",
                ),
                width=3,
            ),
        ],
        className="mb-2",
    )

    # Interactive Filter Bar (drives live callback on Tab 1 & Tab 4)
    filter_bar = dbc.Card(
        dbc.CardBody(
            dbc.Row(
                [
                    dbc.Col(
                        [
                            html.Label("Year Range Filter", className="fw-bold small text-secondary"),
                            dcc.RangeSlider(
                                id="year-range-slider",
                                min=min_yr,
                                max=max_yr,
                                step=1,
                                value=[min_yr, max_yr],
                                marks={y: str(y) for y in range(min_yr, max_yr + 1, 10)},
                                tooltip={"placement": "bottom", "always_visible": False},
                            ),
                        ],
                        width=5,
                    ),
                    dbc.Col(
                        [
                            html.Label("Filter by Country (Top 15)", className="fw-bold small text-secondary"),
                            dcc.Dropdown(
                                id="country-filter",
                                options=[{"label": c, "value": c} for c in top_countries],
                                multi=True,
                                placeholder="All countries",
                            ),
                        ],
                        width=4,
                    ),
                    dbc.Col(
                        [
                            html.Label("Filter by Activity", className="fw-bold small text-secondary"),
                            dcc.Dropdown(
                                id="activity-filter",
                                options=[{"label": a, "value": a} for a in activities],
                                multi=True,
                                placeholder="All activities",
                            ),
                        ],
                        width=3,
                    ),
                ]
            )
        ),
        className="mb-3 shadow-sm",
    )

    # 5 Organized Tabs matching Queries 1-4 + External Tourism Bonus
    tabs = dcc.Tabs(
        [
            dcc.Tab(
                label="1. Grouping & Fatality Rates",
                children=[
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Query 1: Incident Distribution & Fatality Rates", className="card-title"),
                                html.P(
                                    "Use the filters above to dynamically slice attacks by period, country, and activity group.",
                                    className="text-muted",
                                ),
                                dcc.Graph(id="live-countries-graph", figure=figures_dict.get("fig_countries", go.Figure())),
                                html.Hr(),
                                dcc.Graph(id="live-activities-graph", figure=figures_dict.get("fig_activities", go.Figure())),
                                html.Hr(),
                                dcc.Graph(figure=figures_dict.get("fig_species", go.Figure())),
                            ]
                        ),
                        className="mt-3 border-0 shadow-sm",
                    )
                ],
            ),
            dcc.Tab(
                label="2. Frequent Patterns (Apriori)",
                children=[
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Query 2: Association Rule Mining (Apriori)", className="card-title"),
                                html.P(
                                    "Discrete co-occurrences of Country, Activity, Type, Season, Time of Day, and Fatality (Lab 9 methodology).",
                                    className="text-muted",
                                ),
                                dcc.Graph(figure=figures_dict.get("fig_sensitivity", go.Figure())),
                                html.Hr(),
                                dcc.Graph(figure=figures_dict.get("fig_rules", go.Figure())),
                            ]
                        ),
                        className="mt-3 border-0 shadow-sm",
                    )
                ],
            ),
            dcc.Tab(
                label="3. Temporal & Prophet Forecast",
                children=[
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Query 3: Hemisphere Seasonality & Prophet Forecasting", className="card-title"),
                                html.P(
                                    "Inverted monthly summer peaks between Northern and Southern hemispheres, 5-year Prophet forecast (95% CI), and monthly decomposition.",
                                    className="text-muted",
                                ),
                                dcc.Graph(figure=figures_dict.get("fig_seasonality", go.Figure())),
                                html.Hr(),
                                dcc.Graph(figure=figures_dict.get("fig_forecast", go.Figure())),
                                html.Hr(),
                                dcc.Graph(figure=figures_dict.get("fig_components", go.Figure())),
                            ]
                        ),
                        className="mt-3 border-0 shadow-sm",
                    )
                ],
            ),
            dcc.Tab(
                label="4. Spatial Hotspots (GeoPandas + K-Means / OPTICS)",
                children=[
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Query 4: Coastal Spatial Hotspots", className="card-title"),
                                html.P(
                                    "GeoPandas spatial join with Natural Earth country polygons combined with K-Means basin partitioning and OPTICS density clustering (Lab 8).",
                                    className="text-muted",
                                ),
                                dcc.Graph(id="live-spatial-map", figure=figures_dict.get("fig_map", go.Figure())),
                                html.Hr(),
                                dcc.Graph(figure=figures_dict.get("fig_optics_map", go.Figure())),
                            ]
                        ),
                        className="mt-3 border-0 shadow-sm",
                    )
                ],
            ),
            dcc.Tab(
                label="5. Tourism Bonus (+1 Extra Point)",
                children=[
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("External Dataset Bonus: World Bank Tourism Correlation", className="card-title"),
                                html.P(
                                    f"Correlating annual attacks with inbound international tourists (Pearson r = {kpis['pearson_r']}) distinguishes raw exposure volume from relative coastal risk.",
                                    className="text-muted",
                                ),
                                dcc.Graph(figure=figures_dict.get("fig_comparison", go.Figure())),
                                html.Hr(),
                                dcc.Graph(figure=figures_dict.get("fig_dual_axis", go.Figure())),
                            ]
                        ),
                        className="mt-3 border-0 shadow-sm",
                    )
                ],
            ),
        ]
    )

    app.layout = dbc.Container([header, kpi_cards, filter_bar, tabs], fluid=True, className="pb-5")

    # Register live interactive callback when cleaned DataFrames are available
    if clean_sharks is not None:
        @app.callback(
            [
                Output("live-countries-graph", "figure"),
                Output("live-activities-graph", "figure"),
                Output("live-spatial-map", "figure"),
            ],
            [
                Input("year-range-slider", "value"),
                Input("country-filter", "value"),
                Input("activity-filter", "value"),
            ],
        )
        # Function for filtering dashboard charts interactively based on user controls
        # Input: year_range (list), selected_countries (list or None), selected_activities (list or None)
        # Output: tuple of 3 updated Plotly figures (countries, activities, spatial map)
        def update_filtered_charts(year_range, selected_countries, selected_activities):
            df_f = clean_sharks[
                (clean_sharks["Year"] >= year_range[0]) & (clean_sharks["Year"] <= year_range[1])
            ].copy()
            if selected_countries:
                df_f = df_f[df_f["Country_clean"].isin(selected_countries)]
            if selected_activities:
                df_f = df_f[df_f["Activity_group"].isin(selected_activities)]

            if len(df_f) == 0:
                empty = go.Figure().update_layout(title="No incidents match current filters", template="plotly_white")
                return empty, empty, empty

            # Updated Countries Bar
            c_stats = df_f.groupby("Country_clean").agg(
                Total_Attacks=("Case Number", "count"),
                Fatal_Attacks=("Fatal_clean", lambda s: (s == "Y").sum()),
            ).reset_index()
            c_stats["Fatality_Rate_Pct"] = ((c_stats["Fatal_Attacks"] / c_stats["Total_Attacks"]) * 100).round(2)
            c_stats = c_stats.sort_values("Total_Attacks", ascending=False).head(10)
            fig_c = px.bar(
                c_stats,
                x="Total_Attacks",
                y="Country_clean",
                orientation="h",
                color="Fatality_Rate_Pct",
                color_continuous_scale="Tealgrn",
                title=f"Top Countries ({year_range[0]}-{year_range[1]}) & Fatality Rate (%)",
            ).update_layout(yaxis={"autorange": "reversed"}, template="plotly_white")

            # Updated Activities Bar
            a_stats = df_f.groupby("Activity_group").agg(
                Total_Attacks=("Case Number", "count"),
                Fatal_Attacks=("Fatal_clean", lambda s: (s == "Y").sum()),
            ).reset_index()
            a_stats["Fatality_Rate_Pct"] = ((a_stats["Fatal_Attacks"] / a_stats["Total_Attacks"]) * 100).round(2)
            a_stats = a_stats.sort_values("Total_Attacks", ascending=False)
            fig_a = px.bar(
                a_stats,
                x="Activity_group",
                y="Total_Attacks",
                color="Fatality_Rate_Pct",
                color_continuous_scale="Reds",
                title=f"Activity Distribution ({year_range[0]}-{year_range[1]}) & Fatality Rate (%)",
            ).update_layout(template="plotly_white")

            # Updated Spatial Map
            if geo_df is not None:
                gdf_f = geo_df[
                    (geo_df["Year"] >= year_range[0]) & (geo_df["Year"] <= year_range[1])
                ].copy()
                if selected_countries:
                    gdf_f = gdf_f[gdf_f["Country_clean"].isin(selected_countries)]
                if selected_activities:
                    gdf_f = gdf_f[gdf_f["Activity_group"].isin(selected_activities)]
                if len(gdf_f) > 0:
                    pts = gdf_f.groupby(["Country_clean", "Area_clean", "latitude", "longitude", "Cluster_Name"]).agg(
                        Incidents=("Case Number", "count")
                    ).reset_index()
                    fig_m = px.scatter_geo(
                        pts,
                        lat="latitude",
                        lon="longitude",
                        size="Incidents",
                        size_max=30,
                        color="Cluster_Name",
                        hover_name="Area_clean",
                        title=f"Filtered Coastal Hotspots ({year_range[0]}-{year_range[1]})",
                        projection="natural earth",
                    )
                    fig_m.update_geos(showcoastlines=True, coastlinecolor="#94A3B8", showland=True, landcolor="#F1F5F9")
                    fig_m.update_layout(template="plotly_white", legend=dict(orientation="h", yanchor="bottom", y=-0.25))
                else:
                    fig_m = go.Figure().update_layout(title="No geocoded incidents match filters", template="plotly_white")
            else:
                fig_m = figures_dict.get("fig_map", go.Figure())

            return fig_c, fig_a, fig_m

    return app


# Function for exporting all dashboard figures and dynamic KPIs to a standalone offline HTML proof file
# Input: figures_dict (dict), kpis (dict), output_path (str)
# Output: output_path (str) of the written HTML file
def export_dashboard_html(figures_dict, kpis, output_path):
    dir_name = os.path.dirname(output_path)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)

    titles = [
        ("Query 1: Top 10 Countries & Fatality Rate", "fig_countries"),
        ("Query 1: Activity Group Distribution & Fatality Rate", "fig_activities"),
        ("Query 1: Identified Shark Species & Fatality Rate", "fig_species"),
        ("Query 2: Apriori Support Sensitivity Curve", "fig_sensitivity"),
        ("Query 2: Association Rules (Support vs. Confidence, Lift)", "fig_rules"),
        ("Query 3: Northern vs. Southern Hemisphere Seasonality", "fig_seasonality"),
        ("Query 3: Annual Prophet 5-Year Forecast (95% CI)", "fig_forecast"),
        ("Query 3: Prophet Monthly Decomposition (Trend & Seasonality)", "fig_components"),
        ("Query 4: Global Coastal Hotspots (K-Means + GeoPandas)", "fig_map"),
        ("Query 4: Density-Based Hotspots (OPTICS Comparison)", "fig_optics_map"),
        ("Bonus (+1 pt): Raw Volume vs. Attacks per 1M Tourists", "fig_comparison"),
        ("Bonus (+1 pt): US Dual-Axis Attacks vs. Tourism Arrivals", "fig_dual_axis"),
    ]

    figs_html = []
    for title, key in titles:
        fig = figures_dict.get(key)
        if fig is not None:
            figs_html.append(
                f"""
            <div style="background: white; border: 1px solid #E2E8F0; border-radius: 8px; padding: 16px; margin-bottom: 24px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
                <h3 style="margin-top:0; color:#0F172A; font-size:18px;">{title}</h3>
                {fig.to_html(full_html=False, include_plotlyjs='cdn')}
            </div>
            """
            )

    html_doc = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>{PROJECT_TITLE} - Dashboard Evidence</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; background: #F8FAFC; margin: 0; padding: 24px; color: #0F172A; }}
        header {{ background: white; border: 1px solid #E2E8F0; border-radius: 8px; padding: 24px; margin-bottom: 24px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }}
        .badge {{ background: #334155; color: white; padding: 6px 12px; border-radius: 4px; font-size: 13px; font-weight: 600; display: inline-block; margin-top: 8px; }}
        .kpi-container {{ display: flex; gap: 16px; margin-bottom: 24px; flex-wrap: wrap; }}
        .kpi-card {{ flex: 1; min-width: 200px; background: white; border: 1px solid #E2E8F0; border-radius: 8px; padding: 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }}
        .kpi-title {{ font-size: 12px; color: #64748B; margin: 0 0 6px 0; text-transform: uppercase; font-weight: 600; }}
        .kpi-val {{ font-size: 26px; font-weight: 700; color: #0F766E; margin: 0 0 4px 0; }}
    </style>
</head>
<body>
    <header>
        <h1 style="margin:0 0 8px 0; font-size:24px;">{PROJECT_TITLE}</h1>
        <p style="margin:0 0 8px 0; color:#64748B;">{DATASET_NAMES}</p>
        <div class="badge">Team Members: {" | ".join(TEAM_MEMBERS)}</div>
    </header>

    <div class="kpi-container">
        <div class="kpi-card">
            <div class="kpi-title">Total Analyzed Incidents</div>
            <div class="kpi-val">{kpis['total_incidents']}</div>
            <small style="color:#64748B;">{kpis['year_window']}</small>
        </div>
        <div class="kpi-card">
            <div class="kpi-title">Overall Fatality Rate</div>
            <div class="kpi-val" style="color:#DC2626;">{kpis['fatality_rate']}</div>
            <small style="color:#64748B;">{kpis['fatal_subtitle']}</small>
        </div>
        <div class="kpi-card">
            <div class="kpi-title">Top Incident Country</div>
            <div class="kpi-val" style="color:#0284C7;">{kpis['top_country']}</div>
            <small style="color:#64748B;">{kpis['top_country_subtitle']}</small>
        </div>
        <div class="kpi-card">
            <div class="kpi-title">Highest Tourism Risk Rate</div>
            <div class="kpi-val" style="color:#D97706;">{kpis['highest_risk_label']}</div>
            <small style="color:#64748B;">{kpis['highest_risk_subtitle']}</small>
        </div>
    </div>

    {''.join(figs_html)}
</body>
</html>
"""
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_doc)
    return output_path

## Step 8: Main Execution Pipeline (`if __name__ == '__main__':`)

Calls all KDD stages sequentially, displays key tables and interactive Plotly visualizations, exports `Laurent_dashboard.html`, and builds the interactive Dash application.

In [9]:
# Main block (mandatory structure per Final_project.pdf page 4)
if __name__ == '__main__':
    # Cross-platform data directory resolution (works on Windows, macOS, Linux, and standalone DVLC)
    repo_data_dir = Path('..') / 'projet' / 'data'
    data_dir = repo_data_dir if repo_data_dir.exists() else Path('data')
    shark_path = data_dir / 'attacks.csv'
    tourism_path = data_dir / 'tourism_arrivals.csv'
    geo_path = data_dir / 'locations_geocoded.csv'
    shp_path = data_dir / 'countries_shp' / 'ne_110m_admin_0_countries.shp'

    raw_sharks = load_shark_data(shark_path)
    raw_tourism = load_tourism_data(tourism_path)
    shark_eda = summarize_dataset(raw_sharks, 'Global Shark Attack (Raw)')
    tourism_eda = summarize_dataset(raw_tourism, 'World Bank Tourism ST.INT.ARVL')
    print(f"Raw Sharks: {shark_eda['n_rows']} rows x {shark_eda['n_cols']} cols")
    print(f"Raw Tourism: {tourism_eda['n_rows']} rows x {tourism_eda['n_cols']} cols")

    clean_sharks = clean_shark_data(raw_sharks)
    merged_tourism = clean_and_merge_tourism(clean_sharks, raw_tourism)
    print(f'Cleaned Sharks (1960-2018): {len(clean_sharks)} records')
    print(f'Merged Tourism (1995-2018): {len(merged_tourism)} country-year rows across {merged_tourism["Country_clean"].nunique()} countries')

    # Step 2: Indicators construction (Queries 1-4 + Tourism Bonus)
    q1 = query_grouping_analysis(clean_sharks, top_n=10)
    display(q1['top_countries'].head(5))
    display(q1['activity_stats'])
    display(q1['species_stats'].head(6))
    q1['fig_countries'].show()
    q1['fig_activities'].show()
    q1['fig_species'].show()

    q2 = query_frequent_patterns(clean_sharks, min_support=0.03, min_lift=1.05)
    print(f"Apriori: {len(q2['frequent_itemsets'])} frequent itemsets, {len(q2['rules'])} rules")
    display(q2['rules'][['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']].head(8))
    q2['fig_sensitivity'].show()
    q2['fig_rules'].show()

    q3 = query_temporal_forecast(clean_sharks, forecast_years=5)
    display(q3['forecast_df'].tail(5))
    q3['fig_seasonality'].show()
    q3['fig_forecast'].show()
    q3['fig_components'].show()

    geo_coords = geocode_coastal_areas(clean_sharks, cache_path=geo_path)
    q4 = query_spatial_clustering(clean_sharks, geo_coords, shp_path=shp_path, n_clusters=6)
    print(f"Spatial coverage: {q4['coverage_pct']}% | GeoPandas sjoin matches: {q4['sjoin_matched']}")
    display(q4['cluster_summary'])
    q4['fig_map'].show()
    q4['fig_optics_map'].show()

    bonus = query_tourism_enrichment(merged_tourism)
    print(f"Tourism Pearson r = {bonus['pearson_corr']:.4f} (p = {bonus['pearson_pval']:.4e}) | Spearman rho = {bonus['spearman_corr']:.4f}")
    display(bonus['country_comparison'])
    bonus['fig_comparison'].show()
    bonus['fig_dual_axis'].show()

    # Step 3: Build interactive Dash dashboard & export offline HTML proof
    dashboard_figures = {
        'fig_countries': q1['fig_countries'],
        'fig_activities': q1['fig_activities'],
        'fig_species': q1['fig_species'],
        'fig_sensitivity': q2['fig_sensitivity'],
        'fig_rules': q2['fig_rules'],
        'fig_seasonality': q3['fig_seasonality'],
        'fig_forecast': q3['fig_forecast'],
        'fig_components': q3['fig_components'],
        'fig_map': q4['fig_map'],
        'fig_optics_map': q4['fig_optics_map'],
        'fig_comparison': bonus['fig_comparison'],
        'fig_dual_axis': bonus['fig_dual_axis'],
    }
    live_kpis = compute_dashboard_kpis(clean_sharks, bonus)
    export_dashboard_html(dashboard_figures, live_kpis, 'Laurent_dashboard.html')
    app = create_dash_app(dashboard_figures, live_kpis, clean_sharks=clean_sharks, geo_df=q4['geo_df'])
    print('Exported Laurent_dashboard.html and initialized Dash app successfully.')
    print('To launch the live server during the demo, uncomment: app.run(debug=False, port=8050)')
    # app.run(debug=False, port=8050)

Raw Sharks: 25723 rows x 24 cols
Raw Tourism: 266 rows x 30 cols


Cleaned Sharks (1960-2018): 4033 records
Merged Tourism (1995-2018): 355 country-year rows across 65 countries


,Country_clean,Total_Attacks,Fatal_Attacks,Non_Fatal_Attacks,Fatality_Rate_Pct
0,United States,1756,63,1544,3.59
1,Australia,704,76,561,10.80
2,South Africa,432,48,336,11.11
3,Brazil,103,31,63,30.10
4,"Bahamas, The",91,4,82,4.40


,Activity_group,Total_Attacks,Fatal_Attacks,Fatality_Rate_Pct
0,Surfing,1257,66,5.25
1,Diving/Snorkeling,775,122,15.74
2,Other,629,109,17.33
3,Swimming,623,138,22.15
4,Fishing,400,37,9.25
5,Wading,241,12,4.98
6,Boating/Kayaking,108,11,10.19


,Species_group,Total_Attacks,Fatal_Attacks,Fatality_Rate_Pct
0,Unidentified,1730,264,15.26
1,Other Named Shark,850,49,5.76
2,White Shark,561,101,18.00
3,Tiger Shark,220,49,22.27
4,Hammerhead/Mako/Whaler,208,7,3.37
5,Reef/Nurse Shark,190,0,0.00


Apriori: 417 frequent itemsets, 2262 rules


... (7 outputs left out)

,Cluster_Name,Total_Incidents,Fatal_Count,Mean_Lat,Mean_Lon,Dominant_Activity,Fatality_Rate_Pct
0,"US Atlantic, Florida & Caribbean (United State...",1591,105,25.31,-77.96,Surfing,6.60
1,Australia & South-West Pacific (Australia / Ne...,797,129,-29.46,154.31,Surfing,16.19
2,Southern Africa & Indian Ocean (South Africa /...,544,94,-30.14,29.60,Surfing,17.28
3,Pacific & US West Coast / Hawaii (United State...,527,38,25.47,-138.05,Surfing,7.21
4,Australia & South-West Pacific (Australia / Ph...,270,65,-8.75,117.80,Diving/Snorkeling,24.07
5,"Europe, Mediterranean & Red Sea (Italy / Egypt...",126,18,38.69,15.16,Diving/Snorkeling,14.29


Tourism Pearson r = 0.5967 (p = 1.3065e-35) | Spearman rho = 0.3423


,Country_clean,Years_Observed,Total_Attacks,Total_Tourist_Arrivals,attacks_per_1M_tourists
0,United States,24,1180,3.039728e+09,0.388
1,Australia,24,433,1.379510e+08,3.139
2,South Africa,24,173,2.277430e+08,0.760
3,Brazil,23,78,1.147470e+08,0.680
4,"Bahamas, The",22,59,1.095840e+08,0.538
5,New Zealand,19,51,4.536700e+07,1.124
6,New Caledonia,16,31,1.693000e+06,18.311
7,Mexico,12,28,1.091718e+09,0.026


Exported Laurent_dashboard.html and initialized Dash app successfully.
To launch the live server during the demo, uncomment: app.run(debug=False, port=8050)
